# EcoSentinel AI — Module B: Water Sample Predictions & Explainable Inference
**Academic ML Project:** Intelligent Waste and Water Monitoring System  
**Module:** Water Intelligence Inference  

### Objectives:
1. Load trained Random Forest model and preprocessor artifacts (`models/water_quality_model.joblib`).
2. Test inference on diverse benchmark environmental scenarios across India.
3. Compute class probabilities, confidence scores, and continuous WQI ratings.
4. Validate explainable limiting factors (specific CPCB criteria violations) and mitigation recommendations.
5. Verify that inference executes in milliseconds without any model retraining.

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd

sys.path.append('..')
from src.water.predict import predict_water, predict_sample_scenarios

print("Inference module loaded successfully.")

## 1. Single Scenario Prediction: Mountain Headwater vs Industrial Drain
We test inference on two contrasting water samples to inspect the response contract.

In [ ]:
# Pristine Sample
pristine_sample = {
    'dissolved_oxygen': 7.6,
    'ph': 7.3,
    'bod': 1.1,
    'conductivity': 88.0,
    'nitrate': 0.12,
    'total_coliform': 18.0,
    'temperature': 16.0,
    'fecal_coliform': 4.0
}

result_clean = predict_water(pristine_sample)
print(f"[Pristine Sample]")
print(f"Predicted Class: {result_clean['predicted_class']} (Confidence: {result_clean['confidence']:.2%})")
print(f"Risk Level:      {result_clean['risk_level']} | WQI: {result_clean['wqi_score']}")
print(f"Limiting Factors:{result_clean['limiting_factors']}")
print(f"Recommendations: {result_clean['recommendations']}")

In [ ]:
# Heavily Contaminated Drain Sample
drain_sample = {
    'dissolved_oxygen': 1.1,
    'ph': 6.2,
    'bod': 32.0,
    'conductivity': 2600.0,
    'nitrate': 16.5,
    'total_coliform': 65000.0,
    'temperature': 32.0,
    'fecal_coliform': 22000.0
}

result_drain = predict_water(drain_sample)
print(f"[Contaminated Drain Sample]")
print(f"Predicted Class: {result_drain['predicted_class']} (Confidence: {result_drain['confidence']:.2%})")
print(f"Risk Level:      {result_drain['risk_level']} | WQI: {result_drain['wqi_score']}")
print("Limiting Factors Identified:")
for lf in result_drain['limiting_factors']:
    print(f"  - {lf['parameter']}: {lf['observed_value']} ({lf['threshold']})")
print("Recommendations:")
for rec in result_drain['recommendations']:
    print(f"  - {rec}")

## 2. Batch Scenario Benchmarking Across Real Indian Water Bodies
We execute the full test scenario suite and display the structured prediction results.

In [ ]:
scenarios_results = predict_sample_scenarios('../outputs/predictions/water_sample_predictions.json')

# Format summary DataFrame
rows = []
for s in scenarios_results:
    p = s['prediction']
    rows.append({
        'Scenario': s['scenario_name'],
        'Location': s['location'],
        'Predicted Class': p['predicted_class'],
        'Confidence': f"{p['confidence']:.1%}",
        'WQI Score': p['wqi_score'],
        'Risk Level': p['risk_level'],
        'Flagged Violations': len(p['limiting_factors'])
    })

pd.DataFrame(rows)